# E1: Supervised 10-Class (L/R) Classifier

In [ ]:
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
print('Setup done')


In [ ]:
import glob, re, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (silhouette_score, adjusted_rand_score, f1_score,
    normalized_mutual_info_score, classification_report, confusion_matrix)
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')


In [ ]:
# ── Cycle extraction (V11 v2 / tsne_v2 style) ────────────────
CONFIG = {
    'FS': 50.0, 'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_SAVGOL_WINDOW': 21, 'CYCLE_SAVGOL_POLYORDER': 3,
    'CYCLE_MIN_PEAK_DEGS': 300.0, 'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0, 'TARGET_LEN': 64, 'MIN_CYCLE_SAMPLES': 10,
}
KNOWN_PATTERNS = {'overhand', 'sneak_overhand', 'underhand', 'sneak_underhand',
                  'dragon_roll', 'underhand_default'}

def extract_signals(df):
    t = df['timestamp_ms'].values / 1000.0
    A = df[['ax_w','ay_w','az_w']].values
    omega = df[['gx','gy','gz']].values * (np.pi / 180.0)
    return t, A, omega

def detect_cycles(t, omega, fs=50.0):
    mag_deg = np.linalg.norm(omega, axis=1) * (180.0 / np.pi)
    n = len(mag_deg)
    if n < 7: return []
    win = int(CONFIG['CYCLE_SAVGOL_WINDOW'])
    if win % 2 == 0: win += 1
    win = max(5, min(win, n if n % 2 == 1 else n - 1))
    poly = max(1, min(int(CONFIG['CYCLE_SAVGOL_POLYORDER']), win - 2))
    mag_s = savgol_filter(mag_deg, window_length=win, polyorder=poly, mode='interp')
    mag_s = savgol_filter(mag_s, window_length=win, polyorder=poly, mode='interp')
    peaks, _ = find_peaks(mag_s, distance=max(1, int(CONFIG['CYCLE_MIN_PERIOD_S'] * fs)),
                          prominence=CONFIG['CYCLE_PROMINENCE_DEGS'],
                          height=CONFIG['CYCLE_MIN_PEAK_DEGS'])
    if len(peaks) == 0: return []
    cycles = []
    for i, p in enumerate(peaks):
        left = 0 if i == 0 else (peaks[i-1] + p) // 2
        right = (len(t)-1) if i == len(peaks)-1 else (p + peaks[i+1]) // 2
        if right > left and (right - left) >= CONFIG['MIN_CYCLE_SAMPLES']:
            cycles.append((left, right))
    return cycles

def pair_cycles(t0, cyc0, t1, cyc1):
    paired0, paired1, used = [], [], set()
    for c0 in cyc0:
        best_i, best_ov = -1, -1.0
        for i, c1 in enumerate(cyc1):
            if i in used: continue
            ov = max(0, min(t0[c0[1]], t1[c1[1]]) - max(t0[c0[0]], t1[c1[0]]))
            if ov > best_ov: best_ov, best_i = ov, i
        if best_i >= 0 and best_ov > 0:
            paired0.append(c0); paired1.append(cyc1[best_i]); used.add(best_i)
    return paired0, paired1

def resample(sig, n):
    if len(sig) < 2:
        return np.zeros(n) if sig.ndim == 1 else np.zeros((n, sig.shape[1]))
    x_old, x_new = np.linspace(0, 1, len(sig)), np.linspace(0, 1, n)
    if sig.ndim == 1: return np.interp(x_new, x_old, sig)
    return np.column_stack([np.interp(x_new, x_old, sig[:, j]) for j in range(sig.shape[1])])

def build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1):
    tl = CONFIG['TARGET_LEN']
    r0 = resample(np.column_stack([A0[s0:e0], om0[s0:e0]]), tl)
    r1 = resample(np.column_stack([A1[s1:e1], om1[s1:e1]]), tl)
    return np.column_stack([r0, r1]).T.astype(np.float32)

# Fine-grained label mapping
FINE_MAP = {
    'UR': 'UR', 'ur': 'UR', 'UR0': 'UR', 'UR-CW': 'UR',
    'UL0': 'UL',
    'OR': 'OR', 'or': 'OR', 'OR2': 'OR', 'OR3': 'OR', 'OR-OSL': 'OR', 'OR/OSL?': 'OR',
    'OL': 'OL', 'ol': 'OL', 'OL2': 'OL',
    'USR': 'USR', 'USL': 'USL',
    'OSR': 'OSR', 'OSL': 'OSL',
    'FB': 'FB', 'fb': 'FB', 'FB2': 'FB', 'BF2': 'BF', 'bf': 'BF',
    'underhand': 'U_generic', 'overhand': 'O_generic',
    'dragon_roll': 'DR_generic',
    'sneak_underhand': 'SU_generic', 'sneak_overhand': 'SO_generic',
    'CW': None, 'cw': None, 'CCW': None, 'ccw': None,
    'idle': None, 'Idle3': None, 'Idle-or-ol?': None, 'VQ5': None, 'vq16': None,
}
_FINE_PREFIX = [
    (re.compile(r'^USR$', re.I), 'USR'), (re.compile(r'^USL$', re.I), 'USL'),
    (re.compile(r'^OSR$', re.I), 'OSR'), (re.compile(r'^OSL$', re.I), 'OSL'),
    (re.compile(r'^UR', re.I), 'UR'), (re.compile(r'^UL', re.I), 'UL'),
    (re.compile(r'^OR', re.I), 'OR'), (re.compile(r'^OL', re.I), 'OL'),
    (re.compile(r'^FB', re.I), 'FB'), (re.compile(r'^BF', re.I), 'BF'),
    (re.compile(r'^CW$', re.I), None), (re.compile(r'^CCW$', re.I), None),
    (re.compile(r'^idle', re.I), None), (re.compile(r'^vq', re.I), None),
]
def map_fine_label(raw):
    raw = raw.strip()
    if raw in FINE_MAP: return FINE_MAP[raw]
    if raw.lower() in FINE_MAP: return FINE_MAP[raw.lower()]
    for pat, c in _FINE_PREFIX:
        if pat.match(raw): return c
    return None

COARSE_MAP = {
    'UR': 'underhand', 'UL': 'underhand', 'U_generic': 'underhand',
    'OR': 'overhand', 'OL': 'overhand', 'O_generic': 'overhand',
    'USR': 'sneak_underhand', 'USL': 'sneak_underhand', 'SU_generic': 'sneak_underhand',
    'OSR': 'sneak_overhand', 'OSL': 'sneak_overhand', 'SO_generic': 'sneak_overhand',
    'FB': 'dragon_roll', 'BF': 'dragon_roll', 'DR_generic': 'dragon_roll',
}
print('Functions defined')


In [ ]:
# ── Load all cycles with fine labels ──────────────────────────
all_matrices = []; all_fine = []; all_coarse = []; all_groups = []

def load_cycles(d0p, d1p, name, fine='unlabeled', windows=None):
    df0, df1 = pd.read_csv(d0p), pd.read_csv(d1p)
    t0, A0, om0 = extract_signals(df0)
    t1, A1, om1 = extract_signals(df1)
    cyc0 = detect_cycles(t0, om0, CONFIG['FS'])
    cyc1 = detect_cycles(t1, om1, CONFIG['FS'])
    p0, p1 = pair_cycles(t0, cyc0, t1, cyc1)
    if windows:
        fp0, fp1 = [], []
        for (s0,e0),(s1,e1) in zip(p0,p1):
            if any(ws <= (t0[s0]+t0[e0])/2 < we for ws, we in windows):
                fp0.append((s0,e0)); fp1.append((s1,e1))
        p0, p1 = fp0, fp1
    grp = name.split('/')[0] if '/' in name else name
    cnt = 0
    for (s0,e0),(s1,e1) in zip(p0,p1):
        cm = build_cycle_matrix(A0, A1, om0, om1, s0, e0, s1, e1)
        all_matrices.append(cm); all_fine.append(fine)
        all_coarse.append(COARSE_MAP.get(fine, fine)); all_groups.append(grp)
        cnt += 1
    return cnt

# Homogeneous
for d0 in sorted(glob.glob(os.path.join(DATA_PROCESSED, '*_device0_processed.csv'))):
    d1 = d0.replace('_device0_', '_device1_')
    if not os.path.exists(d1): continue
    stem = os.path.basename(d0).replace('_device0_processed.csv', '')
    parts = stem.split('_')
    if len(parts) < 3: continue
    rest = '_'.join(parts[2:])
    fine = 'unlabeled'
    for pat in sorted(KNOWN_PATTERNS, key=len, reverse=True):
        if rest.startswith(pat):
            if pat in ('underhand','underhand_default'): fine = 'U_generic'
            elif pat == 'overhand': fine = 'O_generic'
            elif pat == 'dragon_roll': fine = 'DR_generic'
            elif pat == 'sneak_underhand': fine = 'SU_generic'
            elif pat == 'sneak_overhand': fine = 'SO_generic'
            break
    n = load_cycles(d0, d1, stem, fine)
    if n > 0: print(f'  {stem:<50s} {fine:<12s} {n:>4d}')

# Heterogeneous
if os.path.isdir(NEW_LABELED_RAW):
    for sname in sorted(os.listdir(NEW_LABELED_RAW)):
        sdir = os.path.join(NEW_LABELED_RAW, sname)
        if not os.path.isdir(sdir): continue
        lpath = None
        for fn in ('labels_corrected.json', 'labels.json'):
            p = os.path.join(sdir, fn)
            if os.path.isfile(p): lpath = p; break
        if not lpath: continue
        d0 = os.path.join(DATA_PROCESSED, sname + '_device0_processed.csv')
        d1 = os.path.join(DATA_PROCESSED, sname + '_device1_processed.csv')
        if not (os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lpath, encoding='utf-8') as f: data = _json.load(f)
        segs = data.get('segments', data) if isinstance(data, dict) else data
        wbl = defaultdict(list)
        for seg in segs:
            fine = map_fine_label(seg.get('label', ''))
            if fine is None: continue
            s, e = seg.get('start'), seg.get('end')
            if s is None: continue
            if e is None: e = s + 2.0
            wbl[fine].append((float(s), float(e)))
        for fine, windows in sorted(wbl.items()):
            n = load_cycles(d0, d1, sname + '/' + fine, fine, windows)
            if n > 0: print(f'  {sname}/{fine:<12s} {"":>33s} {n:>4d}')

X_raw = np.array([cm.ravel() for cm in all_matrices])
y_fine = np.array(all_fine)
y_coarse = np.array(all_coarse)
g_all = np.array(all_groups)

# Masks
LR_CLASSES = {'UR','UL','OR','OL','USR','USL','OSR','OSL','FB','BF'}
GENERIC_CLASSES = {'U_generic','O_generic','DR_generic','SU_generic','SO_generic'}
ALL_10_CLASSES = LR_CLASSES

lr_mask = np.array([l in LR_CLASSES for l in y_fine])
labeled_mask = np.array([l != 'unlabeled' for l in y_fine])
coarse_labeled_mask = np.array([l not in ('unlabeled',) for l in y_coarse])

print('\nTotal: ' + str(len(X_raw)) + ' cycles')
print('L/R labeled: ' + str(lr_mask.sum()))
print('Generic labeled: ' + str((labeled_mask & ~lr_mask).sum()))
print('Unlabeled: ' + str((~labeled_mask).sum()))
print('\nFine label distribution:')
for lab, cnt in sorted(Counter(y_fine[labeled_mask]).items(), key=lambda x: -x[1]):
    print(f'  {lab:<12s}: {cnt:>5d}')


In [ ]:
# ── LOSO helper ──────────────────────────────────────────────
def run_loso(X, y, groups, make_clf, preprocess=None):
    unique_groups = np.unique(groups)
    class_groups = defaultdict(set)
    for label, grp in zip(y, groups): class_groups[label].add(grp)
    singleton_classes = {c for c, gs in class_groups.items() if len(gs) < 2}
    test_groups = [g for g in unique_groups
                   if not set(y[groups == g]).issubset(singleton_classes)]
    all_true, all_pred = [], []
    for g in test_groups:
        te = groups == g; tr = ~te
        X_tr, X_te, y_tr, y_te = X[tr], X[te], y[tr], y[te]
        if preprocess:
            X_tr, X_te = preprocess(X_tr, X_te)
        clf = make_clf()
        clf.fit(X_tr, y_tr)
        all_true.extend(y_te.tolist()); all_pred.extend(clf.predict(X_te).tolist())
    all_true, all_pred = np.array(all_true), np.array(all_pred)
    labels = sorted(set(all_true) | set(all_pred))
    return {
        'f1_macro': f1_score(all_true, all_pred, average='macro', labels=labels, zero_division=0),
        'f1_weighted': f1_score(all_true, all_pred, average='weighted', labels=labels, zero_division=0),
        'accuracy': np.mean(all_true == all_pred),
        'cm': confusion_matrix(all_true, all_pred, labels=labels),
        'labels': labels,
        'report': classification_report(all_true, all_pred, labels=labels, zero_division=0),
        'all_true': all_true, 'all_pred': all_pred,
    }

def plot_cm(cm, labels, title):
    fig, ax = plt.subplots(figsize=(max(7, len(labels)*0.9), max(6, len(labels)*0.8)))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    color='white' if cm[i,j] > cm.max()*0.5 else 'black', fontsize=7)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    plt.colorbar(im, ax=ax); plt.tight_layout(); plt.show()

def pp_scale(X_tr, X_te):
    sc = StandardScaler(); return sc.fit_transform(X_tr), sc.transform(X_te)

print('LOSO helper ready')


## E1: Supervised classifier with 10 fine-grained L/R classes

Train on UR, UL, OR, OL, USR, USL, OSR, OSL, FB, BF instead of 5 merged classes.
Generic-labeled cycles (U_generic, O_generic) are excluded since we do not know their L/R.

In [ ]:
# Only use cycles with L/R labels (not generic)
X_lr = X_raw[lr_mask]
y_lr = y_fine[lr_mask]
g_lr = g_all[lr_mask]

print(f'L/R labeled cycles: {len(X_lr)}')
print(f'Groups: {len(np.unique(g_lr))}')
print(f'Classes (10): {sorted(set(y_lr))}')
print(f'Distribution:')
for lab, cnt in sorted(Counter(y_lr).items(), key=lambda x: -x[1]):
    print(f'  {lab:<6s}: {cnt:>5d}')


In [ ]:
def make_rf():
    return RandomForestClassifier(n_estimators=400, class_weight='balanced', random_state=42)
def make_gbm():
    return GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.08, random_state=42)

# ── 10-class RF on raw 768D ──
print('LOSO: 10-class RF on raw 768D...')
res_10_rf = run_loso(X_lr, y_lr, g_lr, make_rf, pp_scale)
print(res_10_rf['report'])
print(f'10-class RF: F1 macro={res_10_rf["f1_macro"]:.3f}, Acc={res_10_rf["accuracy"]:.3f}')

# ── 10-class GBM on raw 768D ──
print('\nLOSO: 10-class GBM on raw 768D...')
res_10_gbm = run_loso(X_lr, y_lr, g_lr, make_gbm, pp_scale)
print(res_10_gbm['report'])
print(f'10-class GBM: F1 macro={res_10_gbm["f1_macro"]:.3f}, Acc={res_10_gbm["accuracy"]:.3f}')


In [ ]:
# ── 5-class baseline on same L/R-labeled data (merge back) ──
y_coarse_lr = np.array([COARSE_MAP[l] for l in y_lr])

print('LOSO: 5-class RF (same data, merged labels)...')
res_5_rf = run_loso(X_lr, y_coarse_lr, g_lr, make_rf, pp_scale)
print(res_5_rf['report'])
print(f'5-class RF: F1 macro={res_5_rf["f1_macro"]:.3f}, Acc={res_5_rf["accuracy"]:.3f}')


In [ ]:
# ── 10-class -> mapped to 5-class for comparison ──
# Predict 10 classes, then merge predictions back to 5 for fair comparison
pred_10 = res_10_rf['all_pred']
true_10 = res_10_rf['all_true']
pred_mapped = np.array([COARSE_MAP[p] for p in pred_10])
true_mapped = np.array([COARSE_MAP[t] for t in true_10])
labels_5 = sorted(set(true_mapped) | set(pred_mapped))
f1_mapped = f1_score(true_mapped, pred_mapped, average='macro', labels=labels_5, zero_division=0)
acc_mapped = np.mean(true_mapped == pred_mapped)
cm_mapped = confusion_matrix(true_mapped, pred_mapped, labels=labels_5)

print(f'10-class RF mapped to 5-class: F1 macro={f1_mapped:.3f}, Acc={acc_mapped:.3f}')
print(classification_report(true_mapped, pred_mapped, labels=labels_5, zero_division=0))


In [ ]:
plot_cm(res_10_rf['cm'], res_10_rf['labels'], f'10-class RF (F1={res_10_rf["f1_macro"]:.3f})')
plot_cm(res_5_rf['cm'], res_5_rf['labels'], f'5-class RF baseline (F1={res_5_rf["f1_macro"]:.3f})')
plot_cm(cm_mapped, labels_5, f'10-class RF mapped to 5 (F1={f1_mapped:.3f})')


In [ ]:
print('='*70)
print('E1: SUPERVISED 10-CLASS SUMMARY')
print('='*70)
print(f'L/R labeled cycles: {len(X_lr)}')
print(f'Groups: {len(np.unique(g_lr))}')
print(f'10 classes: {sorted(set(y_lr))}')
print(f'Distribution: {dict(Counter(y_lr))}')
print(f'')
print(f'10-class RF raw 768D:      F1 macro={res_10_rf["f1_macro"]:.3f}, Acc={res_10_rf["accuracy"]:.3f}')
print(f'10-class GBM raw 768D:     F1 macro={res_10_gbm["f1_macro"]:.3f}, Acc={res_10_gbm["accuracy"]:.3f}')
print(f'5-class RF (same data):    F1 macro={res_5_rf["f1_macro"]:.3f}, Acc={res_5_rf["accuracy"]:.3f}')
print(f'10->5 mapped RF:           F1 macro={f1_mapped:.3f}, Acc={acc_mapped:.3f}')
print(f'V08 reference:             F1 macro=0.632')
print(f'')
print(f'Does 10-class improve 5-class at the 5-class level?')
print(f'  Direct 5-class:  {res_5_rf["f1_macro"]:.3f}')
print(f'  10->5 mapped:    {f1_mapped:.3f}')
print(f'  Delta:           {f1_mapped - res_5_rf["f1_macro"]:+.3f}')
print(f'')
print(f'Per-class F1 (10-class):')
for lab in sorted(set(y_lr)):
    mask_t = res_10_rf['all_true'] == lab
    if mask_t.sum() == 0: continue
    correct = (res_10_rf['all_pred'][mask_t] == lab).sum()
    print(f'  {lab:<6s}: {correct}/{mask_t.sum()} correct')
